# Experiment 1 — Semantic / Phonetic Probing

*Paper Sec 2.3. Reproduces Fig 2.*

For each codec we extract per-layer accumulated decoded features, slice each MFA-aligned word from its parent utterance, time-pool the slice, and compute the **Euclidean distance** between paired words. Three pair groups: WordNet **synonym**, near-homophone (CMU-dict phoneme Levenshtein < 0.4), and **random** baseline. The expected outcome (paper Fig 2 bottom row) is that synonym distances *exceed* near-homophone distances at most layers — i.e. codec features encode phonetic similarity more strongly than lexical-semantic similarity.

Prerequisite: run `python scripts/prepare_librispeech.py …` first to produce `librispeech.df.pkl` and `librispeech.wordmap.pkl`.

In [ ]:
import os, sys, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load DataFrame + pair maps

In [ ]:
WORDPAIRS_DIR = os.path.join(REPO_ROOT, 'data', 'word_pairs')
df = pd.read_pickle(os.path.join(WORDPAIRS_DIR, 'librispeech.df.pkl'))
with open(os.path.join(WORDPAIRS_DIR, 'librispeech.wordmap.pkl'), 'rb') as f:
    maps = pickle.load(f)
synonym_map, homophone_map = maps['synonym_map'], maps['homophone_map']
print(f'{len(df):,} word occurrences across {df.text.nunique():,} unique words')
print(f'words with synonyms:  {len(synonym_map):,}')
print(f'words with homophones:{len(homophone_map):,}')
df.head(3)

## 2. Sample pairs

``N_PAIRS`` controls all three pair groups. Use a small number while iterating; raise to 5–10k for the final figure.

In [ ]:
from src.data import sample_pairs_from_map, sample_random_pairs

N_PAIRS = 2000
SEED = 0
syn_pairs  = sample_pairs_from_map(df, synonym_map,   N_PAIRS, seed=SEED)
hom_pairs  = sample_pairs_from_map(df, homophone_map, N_PAIRS, seed=SEED + 1)
rand_pairs = sample_random_pairs(df, N_PAIRS, seed=SEED + 2)
PAIRS = {'synonym': syn_pairs, 'homophone': hom_pairs, 'random': rand_pairs}
{k: len(v) for k, v in PAIRS.items()}

## 3. Encode once per audio file, then slice + pool per word

For each codec, we (a) encode every unique `path` in the DataFrame just once, then (b) for each row, slice the layered features by the word's `[start, finish]` on the codec's own frame grid and pool over time. This matches `extract_features.py` in juice500ml's repo, generalized to neural codecs.

In [ ]:
from src.codecs import load_encodec, load_dac, load_mimi, load_mimo
from src.analysis import extract_codec_features, pair_layer_distances

CODEC_FACTORIES = {
    'encodec': lambda: load_encodec(bandwidth=12.0, device=DEVICE),
    'dac':     lambda: load_dac(device=DEVICE),
    'mimi':    lambda: load_mimi(device=DEVICE),
    'mimo':    lambda: load_mimo(checkpoint='/data/xuanshi/REPO/vocal_tract_distance/lib/MiMo_Audio_Tokenizer/MiMo-Audio-Tokenizer', device=DEVICE),
}
RUN_CODECS = ['encodec', 'mimi']  # adjust to taste; full = ['encodec', 'dac', 'mimi', 'mimo']
POOL = 'mean'

In [ ]:
cache_dir = os.path.join(REPO_ROOT, 'data', 'euclidean_cache')
os.makedirs(cache_dir, exist_ok=True)
# Restrict df to rows actually referenced by any pair (saves a lot of encoding)
used = sorted({i for pairs in PAIRS.values() for ab in pairs for i in ab})
df_used = df.loc[used].copy()
print(f'encoding {df_used.path.nunique():,} unique audio paths for {len(df_used):,} word slots')
results = {}
for name in RUN_CODECS:
    try:
        codec = CODEC_FACTORIES[name]()
    except Exception as e:
        print(f'  skipping {name}: {e}'); continue
    df_feat = extract_codec_features(codec, df_used, pool=POOL)
    group_dists = {tag: pair_layer_distances(df_feat, pairs) for tag, pairs in PAIRS.items()}
    results[name] = group_dists
    with open(os.path.join(cache_dir, f'{name}.pkl'), 'wb') as f:
        pickle.dump(group_dists, f)
    del codec, df_feat; torch.cuda.empty_cache()
print('done:', list(results))

## 4. Plot Fig 2 — absolute & normalized distances per layer

In [ ]:
def plot_codec(ax_top, ax_bot, name, group_dists):
    baseline = group_dists['random'].mean(axis=0)
    for tag in ['synonym', 'homophone', 'random']:
        ax_top.plot(group_dists[tag].mean(axis=0), marker='o', label=tag)
    ax_top.set_title(name); ax_top.legend(); ax_top.set_xlabel('layer'); ax_top.set_ylabel('Euclidean')
    for tag in ['synonym', 'homophone']:
        ax_bot.plot(group_dists[tag].mean(axis=0) - baseline, marker='o', label=tag)
    ax_bot.axhline(0, color='gray', lw=0.5)
    ax_bot.set_title(f'{name} (− random)'); ax_bot.legend(); ax_bot.set_xlabel('layer')

n = max(len(results), 1)
fig, axes = plt.subplots(2, n, figsize=(4*n, 6), squeeze=False)
for i, (name, gd) in enumerate(results.items()):
    plot_codec(axes[0, i], axes[1, i], name, gd)
plt.tight_layout(); plt.show()